In [1]:
import torch, torch.nn as nn, random
import numpy as np
torch.manual_seed(0); random.seed(0); np.random.seed(0)

# vocab: 0 = EOS/pad, 1..10 = numbers, 11 = GO
V, E, H, Z = 12, 32, 128, 48
EVENS = [2,4,6,8,10]          # the schema (canonical order)
ODDS  = [1,3,5,7,9]           # fillers, no structure
MAXLEN = 12

def make_seq():
    # evens: 80% canonical order, 20% scrambled (so encoder CAN represent deviations)
    ev = EVENS[:] if random.random() < 0.5 else random.sample(EVENS, len(EVENS))
    k = random.randint(0, 4)                      # a few random odd fillers
    od = [random.choice(ODDS) for _ in range(k)]
    # interleave: evens keep their relative order, odds dropped in at random spots
    seq = ev[:]
    for o in od:
        seq.insert(random.randint(0, len(seq)), o)
    return seq[:MAXLEN]

def to_tensor(seqs):
    inp = torch.zeros(len(seqs), MAXLEN, dtype=torch.long)
    tin = torch.full((len(seqs), MAXLEN), 11, dtype=torch.long)   # decoder input starts w/ GO
    tgt = torch.zeros(len(seqs), MAXLEN, dtype=torch.long)
    for i, s in enumerate(seqs):
        L = len(s)
        inp[i,:L] = torch.tensor(s)
        tin[i,1:L+1 if L+1<=MAXLEN else MAXLEN] = torch.tensor(s[:MAXLEN-1])
        tgt[i,:L] = torch.tensor(s)                                # rest stays 0 = EOS
    return inp, tin, tgt

class AE(nn.Module):
    def __init__(s):
        super().__init__()
        s.emb=nn.Embedding(V,E); s.enc=nn.GRU(E,H,batch_first=True)
        s.toz=nn.Linear(H,Z); s.fromz=nn.Linear(Z,H)
        s.dec=nn.GRU(E,H,batch_first=True); s.out=nn.Linear(H,V)
    def encode(s,inp):
        _,h=s.enc(s.emb(inp)); return s.toz(h[-1])
    def decode(s,z,tin):
        h=s.fromz(z).unsqueeze(0); o,_=s.dec(s.emb(tin),h); return s.out(o)
    def forward(s,inp,tin,sigma):
        z=s.encode(inp); z=z+torch.randn_like(z)*sigma
        return s.decode(z,tin)

m=AE(); opt=torch.optim.Adam(m.parameters(),1e-3); lossf=nn.CrossEntropyLoss()

# ---- train (denoising: small noise so the bottleneck is used + robust) ----
for epoch in range(300):
    seqs=[make_seq() for _ in range(512)]
    inp,tin,tgt=to_tensor(seqs)
    logits=m(inp,tin,sigma=0.2)
    loss=lossf(logits.reshape(-1,V),tgt.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()
print(f"final train loss {loss.item():.3f}\n")

# ---- greedy recall at a given noise level ----
def recall(seq,sigma):
    m.eval()
    with torch.no_grad():
        inp,_,_=to_tensor([seq]); z=m.encode(inp)+torch.randn(1,Z)*sigma
        h=m.fromz(z).unsqueeze(0); tok=torch.tensor([[11]]); out=[]
        for _ in range(MAXLEN):
            o,h=m.dec(m.emb(tok),h); nxt=m.out(o)[0,-1].argmax().item()
            if nxt==0: break
            out.append(nxt); tok=torch.tensor([[nxt]])
    return out

# ---- TEST on a scrambled input (evens out of order + odd fillers) ----
test=[3,8,5,2,9,6,1,10,4]      # evens present but scrambled: 8,2,6,10,4
print("INPUT:",test)
print("(schema canonical evens = 2,4,6,8,10)\n")
for sigma in [0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 5.0]:
    # average over a few noise draws to see the typical pattern
    outs=[recall(test,sigma) for _ in range(7)]
    [print(f"sigma={sigma:>4}:", o) for o in outs[:3]]; print()

/home/nuttidalab/miniconda3/envs/narrative_map/lib/python3.11/site-packages/torch/autograd/graph.py:869: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


final train loss 0.557

INPUT: [3, 8, 5, 2, 9, 6, 1, 10, 4]
(schema canonical evens = 2,4,6,8,10)

sigma= 0.0: [7, 1, 2, 8, 6, 1, 10, 9, 4]
sigma= 0.0: [7, 1, 2, 8, 6, 1, 10, 9, 4]
sigma= 0.0: [7, 1, 2, 8, 6, 1, 10, 9, 4]

sigma= 0.5: [1, 1, 1, 2, 8, 6, 10, 9, 9]
sigma= 0.5: [1, 1, 2, 8, 6, 7, 10, 9, 4]
sigma= 0.5: [7, 1, 2, 8, 4, 6, 10, 1]

sigma= 1.0: [7, 7, 2, 4, 6, 8, 1, 10]
sigma= 1.0: [7, 1, 8, 2, 4, 6, 10, 1]
sigma= 1.0: [1, 1, 6, 8, 2, 4, 1, 10, 1]

sigma= 1.5: [1, 2, 1, 8, 6, 4, 10, 1]
sigma= 1.5: [9, 9, 2, 7, 7, 10, 6, 8, 9]
sigma= 1.5: [9, 8, 1, 2, 9, 10, 7, 6, 7]

sigma= 2.0: [2, 9, 4, 6, 7, 8, 7, 10]
sigma= 2.0: [9, 3, 4, 6, 8, 3, 10, 2, 9]
sigma= 2.0: [9, 8, 1, 4, 6, 2, 3, 10]

sigma= 3.0: [1, 2, 1, 6, 1, 8, 1, 10]
sigma= 3.0: [9, 6, 4, 3, 6, 8, 10, 2]
sigma= 3.0: [2, 9, 9, 4, 9, 6, 8, 9]

sigma= 5.0: [1, 1, 2, 7, 7, 6, 8, 7, 10, 1, 1, 1]
sigma= 5.0: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 6, 8]
sigma= 5.0: [8, 5, 8, 5, 5, 10, 5, 5, 8, 9, 10, 9]

